# Notebook 12: Training Phase 2 - LoRA Fine-tuning

## Objectif
Deuxième phase d'entraînement du modèle Qwen2.5-VL-7B pour CassavaVLM:
- **Phase 2**: Fine-tuning complet avec LoRA adapters (2-3 epochs, ~3-6h)
- Charger le checkpoint Phase 1 (projecteur aligné)
- Dégeler les LoRA adapters + projecteur
- Early stopping pour éviter overfitting
- Target: 95-97% accuracy

## Configuration Hardware
- **GPU**: NVIDIA RTX 5090 32GB VRAM
- **Mixed Precision**: bfloat16 (bf16)
- **Gradient Checkpointing**: Activé pour économiser VRAM

## Inputs
- Checkpoint Phase 1: `models/checkpoints/phase1_projector/best/`
- LoRA config: du Notebook 09
- Class weights: du Notebook 10 (16.7:1 compensated)
- Dataset: 30,491 images JSONL format

## Outputs
- Checkpoints Phase 2: `models/checkpoints/phase2_lora/`
- Logs TensorBoard: `outputs/logs/phase2/`
- Métriques WandB: Online dashboard
- Métriques finales: `outputs/phase2_metrics.json`

## Durée Estimée
**~3-6 heures** sur RTX 5090 (2-3 epochs, 24,392 images train)

## 1. Imports & Setup

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Transformers & PEFT
from transformers import AutoModelForVision2Seq, AutoProcessor
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# WandB (optionnel)
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("⚠️ WandB non disponible, utilisation de TensorBoard uniquement")

# Configuration
PROJECT_ROOT = Path("/home/hounfodjidagba/learning/cassava")
VLM_DATASET_DIR = PROJECT_ROOT / "outputs" / "vlm_dataset"
WEIGHTS_DIR = PROJECT_ROOT / "outputs" / "augmentation_weights"
PHASE1_CHECKPOINT = PROJECT_ROOT / "models" / "checkpoints" / "phase1_projector" / "best"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints" / "phase2_lora"
LOGS_DIR = PROJECT_ROOT / "outputs" / "logs" / "phase2"
METRICS_DIR = PROJECT_ROOT / "outputs"

# Créer les répertoires
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*70)
print("CONFIGURATION TRAINING PHASE 2")
print("="*70)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"\nDirectories:")
print(f"  Phase 1 Checkpoint: {PHASE1_CHECKPOINT}")
print(f"  Dataset: {VLM_DATASET_DIR}")
print(f"  Phase 2 Checkpoints: {CHECKPOINT_DIR}")
print(f"  Logs: {LOGS_DIR}")
print("="*70)

CONFIGURATION TRAINING PHASE 2
Device: cuda
GPU: NVIDIA GeForce RTX 5090
VRAM: 33.67 GB
PyTorch version: 2.9.1+cu128
CUDA version: 12.8

Directories:
  Phase 1 Checkpoint: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector/best
  Dataset: /home/hounfodjidagba/learning/cassava/outputs/vlm_dataset
  Phase 2 Checkpoints: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase2_lora
  Logs: /home/hounfodjidagba/learning/cassava/outputs/logs/phase2


## 2. Load Configurations

In [2]:
# Charger la config LoRA du Notebook 09
lora_config_path = PROJECT_ROOT / "outputs" / "lora_dataloader" / "lora_dataloader_config.json"
with open(lora_config_path, 'r') as f:
    config_data = json.load(f)
    lora_config_dict = config_data['lora_config']

print("="*70)
print("CONFIGURATION LORA (du Notebook 09)")
print("="*70)
print(f"Rank (r): {lora_config_dict['r']}")
print(f"Alpha: {lora_config_dict['lora_alpha']}")
print(f"Dropout: {lora_config_dict['lora_dropout']}")
print(f"Target modules: {lora_config_dict['target_modules']}")
print(f"Task type: {lora_config_dict['task_type']}")
print("="*70)

# Charger les class weights du Notebook 10
class_weights_path = WEIGHTS_DIR / "class_weights.pt"
class_weights = torch.load(class_weights_path)

print("\n" + "="*70)
print("CLASS WEIGHTS (du Notebook 10)")
print("="*70)
print(f"Class weights tensor: {class_weights}")
print(f"Shape: {class_weights.shape}")
print(f"Ratio max/min: {class_weights.max() / class_weights.min():.1f}:1")
print("  → Compense le déséquilibre 16.7:1 des classes")
print("="*70)

# Charger dataset info
dataset_info_path = VLM_DATASET_DIR / "dataset_info.json"
with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

print("\n" + "="*70)
print("DATASET INFO")
print("="*70)
print(f"Total images: {dataset_info['total_images']:,}")
print(f"Train split: {dataset_info['splits']['train']} images")
print(f"Val split: {dataset_info['splits']['val']} images")
print(f"Test split: {dataset_info['splits']['test']} images")
print("="*70)

CONFIGURATION LORA (du Notebook 09)
Rank (r): 16
Alpha: 32
Dropout: 0.05
Target modules: ['o_proj', 'k_proj', 'q_proj', 'v_proj']
Task type: TaskType.CAUSAL_LM

CLASS WEIGHTS (du Notebook 10)
Class weights tensor: tensor([5.6101, 1.1759, 2.5558, 0.3362, 1.6499])
Shape: torch.Size([5])
Ratio max/min: 16.7:1
  → Compense le déséquilibre 16.7:1 des classes

DATASET INFO
Total images: 30,491
Train split: {'num_samples': 24392, 'percentage': 79.99737627496638} images
Val split: {'num_samples': 3049, 'percentage': 9.999672034370798} images
Test split: {'num_samples': 3050, 'percentage': 10.002951690662819} images


## 3. Load Phase 1 Checkpoint

Charger le modèle avec projecteur aligné de Phase 1.

In [3]:
print("Chargement du checkpoint Phase 1...")
print(f"Checkpoint path: {PHASE1_CHECKPOINT}")

# Charger le modèle depuis le checkpoint Phase 1
model = AutoModelForVision2Seq.from_pretrained(
    PHASE1_CHECKPOINT,
    dtype="auto",  # Utilise bfloat16 automatiquement sur RTX 5090
    device_map="auto"
)

# Charger le processor
processor = AutoProcessor.from_pretrained(PHASE1_CHECKPOINT)

print("\n✓ Modèle Phase 1 chargé (projecteur aligné)")

# Activer gradient checkpointing pour économiser VRAM
model.gradient_checkpointing_enable()
print("✓ Gradient checkpointing activé")

# Charger le training state de Phase 1 (sur CPU pour économiser VRAM)
phase1_state_path = PHASE1_CHECKPOINT / "training_state.pt"
if phase1_state_path.exists():
    # IMPORTANT: map_location='cpu' pour ne pas surcharger la VRAM
    # On n'a besoin que des métriques, pas du model_state_dict
    phase1_state = torch.load(phase1_state_path, map_location='cpu')
    print("\n" + "="*70)
    print("RÉSULTATS PHASE 1")
    print("="*70)
    print(f"Epoch: {phase1_state['epoch'] + 1}")
    print(f"Train Loss: {phase1_state['train_loss']:.4f}")
    print(f"Val Loss: {phase1_state['val_loss']:.4f}")
    print("="*70)
    
    # Libérer la mémoire immédiatement
    del phase1_state
    import gc
    gc.collect()
else:
    print("⚠️ training_state.pt non trouvé, continuons sans les métriques Phase 1")

Chargement du checkpoint Phase 1...
Checkpoint path: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector/best


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]


✓ Modèle Phase 1 chargé (projecteur aligné)
✓ Gradient checkpointing activé

RÉSULTATS PHASE 1
Epoch: 1
Train Loss: 6.5779
Val Loss: 6.5445


## 4. Update Freezing Strategy - Phase 2

**Phase 2**: Dégeler les LoRA adapters + garder le projecteur entraînable
- ✅ Entraîner: LoRA adapters (~10M params)
- ✅ Entraîner: Projecteur (~519M params)
- ❌ Geler: Vision encoder base
- ❌ Geler: LLM base

In [4]:
print("="*70)
print("FREEZING STRATEGY - PHASE 2")
print("="*70)

# Pour Phase 2, on dégèle les LoRA adapters
for name, param in model.named_parameters():
    # Dégeler les LoRA adapters
    if 'lora_' in name:
        param.requires_grad = True
    # Garder le projecteur entraînable
    elif 'visual' in name and ('proj' in name or 'merge' in name):
        param.requires_grad = True
    # Geler le reste
    else:
        param.requires_grad = False

# Compter les paramètres entraînables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nParamètres entraînables: {trainable_params:,}")
print(f"Paramètres totaux: {total_params:,}")
print(f"Pourcentage entraînable: {100 * trainable_params / total_params:.4f}%")

# Afficher les modules entraînables
print("\nModules entraînables:")
trainable_modules = [name for name, param in model.named_parameters() if param.requires_grad]

# Compter par catégorie
lora_modules = [name for name in trainable_modules if 'lora_' in name]
proj_modules = [name for name in trainable_modules if 'proj' in name or 'merge' in name]

print(f"  LoRA adapters: {len(lora_modules)} modules")
print(f"  Projecteur: {len(proj_modules)} modules")

print("\n⚠️ PHASE 2: Entraînement LoRA + Projecteur")
print("   → Fine-tuning complet pour classification haute précision")
print("="*70)

FREEZING STRATEGY - PHASE 2

Paramètres entraînables: 529,151,488
Paramètres totaux: 8,302,259,200
Pourcentage entraînable: 6.3736%

Modules entraînables:
  LoRA adapters: 224 modules
  Projecteur: 486 modules

⚠️ PHASE 2: Entraînement LoRA + Projecteur
   → Fine-tuning complet pour classification haute précision


## 5. Training Configuration Phase 2

In [5]:
# Hyperparamètres Phase 2
training_config = {
    # Training
    'num_epochs': 3,  # 2-3 epochs
    'batch_size': 2,  # Petit batch pour VRAM
    'gradient_accumulation_steps': 8,  # Batch effectif = 16
    'learning_rate': 5e-5,  # LR plus bas que Phase 1 (1e-3)
    'weight_decay': 0.01,
    'warmup_ratio': 0.05,  # 5% warmup
    'max_grad_norm': 1.0,  # Gradient clipping
    
    # Optimizer
    'optimizer': 'AdamW',
    'betas': (0.9, 0.999),
    'eps': 1e-8,
    
    # Scheduler
    'scheduler': 'cosine',
    
    # Early Stopping
    'early_stopping_patience': 2,
    'early_stopping_threshold': 0.001,
    
    # Mixed precision
    'mixed_precision': 'bf16',  # bfloat16 pour RTX 5090
    
    # Logging
    'log_every_n_steps': 10,
    'eval_every_n_steps': 500,
    'save_every_n_steps': 1000,
    
    # Other
    'seed': 42,
}

print("="*70)
print("HYPERPARAMÈTRES PHASE 2")
print("="*70)
for key, value in training_config.items():
    print(f"{key:30s}: {value}")
print("="*70)

# Set seed
torch.manual_seed(training_config['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(training_config['seed'])

HYPERPARAMÈTRES PHASE 2
num_epochs                    : 3
batch_size                    : 2
gradient_accumulation_steps   : 8
learning_rate                 : 5e-05
weight_decay                  : 0.01
warmup_ratio                  : 0.05
max_grad_norm                 : 1.0
optimizer                     : AdamW
betas                         : (0.9, 0.999)
eps                           : 1e-08
scheduler                     : cosine
early_stopping_patience       : 2
early_stopping_threshold      : 0.001
mixed_precision               : bf16
log_every_n_steps             : 10
eval_every_n_steps            : 500
save_every_n_steps            : 1000
seed                          : 42


## 6. Early Stopping Class

In [6]:
class EarlyStopping:
    """Early stopping pour arrêter l'entraînement quand val_loss ne s'améliore plus."""
    
    def __init__(self, patience=2, threshold=0.001, verbose=True):
        """
        Args:
            patience: Nombre d'epochs sans amélioration avant d'arrêter
            threshold: Amélioration minimale requise pour être considérée comme significative
            verbose: Afficher les messages
        """
        self.patience = patience
        self.threshold = threshold
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, val_loss, epoch):
        """
        Args:
            val_loss: Validation loss de l'epoch actuelle
            epoch: Numéro de l'epoch actuelle
        """
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_epoch = epoch
            if self.verbose:
                print(f"✓ Early Stopping: Baseline set to {val_loss:.4f}")
        elif val_loss > self.best_loss - self.threshold:
            # Pas d'amélioration significative
            self.counter += 1
            if self.verbose:
                print(f"⚠️ Early Stopping: No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print(f"⛔ Early Stopping: Triggered! Best was epoch {self.best_epoch + 1} with loss {self.best_loss:.4f}")
        else:
            # Amélioration significative
            improvement = self.best_loss - val_loss
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            if self.verbose:
                print(f"✓ Early Stopping: Improvement of {improvement:.4f}, counter reset")

print("✓ Classe EarlyStopping définie")

✓ Classe EarlyStopping définie


## 7. Dataset & DataLoaders

Réutiliser le Dataset du Notebook 11.

In [7]:
# Importer le dataset du notebook 11
from PIL import Image
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class CassavaVLMDataset(Dataset):
    """Dataset pour Qwen2.5-VL avec format JSONL."""
    
    def __init__(self, jsonl_path, image_base_path, processor, max_length=512):
        self.samples = []
        self.image_base_path = Path(image_base_path)
        self.processor = processor
        self.max_length = max_length
        
        # Charger les samples du JSONL
        with open(jsonl_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Charger l'image
        image_path = self.image_base_path / sample['image']
        image = Image.open(image_path).convert('RGB')
        
        # Construire les messages au format Qwen
        messages = []
        for conv in sample['conversations']:
            role = "user" if conv['from'] == "human" else "assistant"
            content_text = conv['value']
            
            if '<image>' in content_text:
                # Message avec image
                text = content_text.replace('<image>\n', '').replace('<image>', '')
                content = [
                    {"type": "image", "image": image},
                    {"type": "text", "text": text}
                ]
            else:
                # Message texte seul
                content = content_text
            
            messages.append({"role": role, "content": content})
        
        # Traiter avec le processor Qwen
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Extraire les tensors
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        pixel_values = inputs.get('pixel_values', None)
        image_grid_thw = inputs.get('image_grid_thw', None)
        
        if pixel_values is not None:
            pixel_values = pixel_values.squeeze(0)
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.squeeze(0)
        
        # Labels = input_ids (autoregressive training)
        labels = input_ids.clone()
        
        # Extraire metadata
        label = sample['metadata']['label']
        class_short = sample['metadata']['class_short']
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': pixel_values,
            'image_grid_thw': image_grid_thw,
            'labels': labels,
            'label': label,
            'class_short': class_short,
            'image_id': sample['id']
        }

def collate_fn(batch):
    """Collate function pour batching."""
    # Trouver les longueurs max
    max_length = max(item['input_ids'].size(0) for item in batch)
    
    # Préparer les tensors
    input_ids = []
    attention_masks = []
    labels_list = []
    pixel_values_list = []
    image_grid_thw_list = []
    label_list = []
    class_short_list = []
    image_id_list = []
    
    for item in batch:
        # Pad input_ids et attention_mask
        seq_len = item['input_ids'].size(0)
        pad_len = max_length - seq_len
        
        input_ids.append(torch.cat([
            item['input_ids'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        attention_masks.append(torch.cat([
            item['attention_mask'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        # Pad labels avec -100 (ignore index)
        labels_list.append(torch.cat([
            item['labels'],
            torch.full((pad_len,), -100, dtype=torch.long)
        ]))
        
        if item['pixel_values'] is not None:
            pixel_values_list.append(item['pixel_values'])
        
        if item['image_grid_thw'] is not None:
            image_grid_thw_list.append(item['image_grid_thw'])
        
        label_list.append(item['label'])
        class_short_list.append(item['class_short'])
        image_id_list.append(item['image_id'])
    
    # Stack tensors
    batched = {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attention_masks),
        'labels': torch.stack(labels_list),
        'label': torch.tensor(label_list, dtype=torch.long),
        'class_short': class_short_list,
        'image_id': image_id_list
    }
    
    if pixel_values_list:
        batched['pixel_values'] = torch.stack(pixel_values_list)
    
    if image_grid_thw_list:
        batched['image_grid_thw'] = torch.stack(image_grid_thw_list)
    
    return batched

print("✓ Classes Dataset et collate_fn définies")

✓ Classes Dataset et collate_fn définies


In [8]:
# Paths des JSONL
train_jsonl = VLM_DATASET_DIR / "train.jsonl"
val_jsonl = VLM_DATASET_DIR / "val.jsonl"

# Créer les datasets
print("Création des datasets...")
train_dataset = CassavaVLMDataset(
    jsonl_path=train_jsonl,
    image_base_path=VLM_DATASET_DIR,
    processor=processor,
    max_length=512
)

val_dataset = CassavaVLMDataset(
    jsonl_path=val_jsonl,
    image_base_path=VLM_DATASET_DIR,
    processor=processor,
    max_length=512
)

# Créer les dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=training_config['batch_size'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=training_config['batch_size'],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

print("="*70)
print("DATALOADERS CRÉÉS")
print("="*70)
print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset: {len(val_dataset):,} samples")
print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")
print(f"Batch size: {training_config['batch_size']}")
print(f"Effective batch size: {training_config['batch_size'] * training_config['gradient_accumulation_steps']}")
print("="*70)

Création des datasets...
DATALOADERS CRÉÉS
Train dataset: 24,392 samples
Val dataset: 3,049 samples
Train batches: 12,196
Val batches: 1,525
Batch size: 2
Effective batch size: 16


## 8. Optimizer & Scheduler

In [9]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Optimizer
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=training_config['learning_rate'],
    betas=training_config['betas'],
    eps=training_config['eps'],
    weight_decay=training_config['weight_decay']
)

# Calculer le nombre total de steps
num_training_steps = len(train_loader) * training_config['num_epochs'] // training_config['gradient_accumulation_steps']
warmup_steps = int(num_training_steps * training_config['warmup_ratio'])

# Scheduler: Warmup + Cosine
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=num_training_steps - warmup_steps,
    eta_min=training_config['learning_rate'] * 0.1
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)

print("="*70)
print("OPTIMIZER & SCHEDULER")
print("="*70)
print(f"Optimizer: AdamW")
print(f"Learning rate: {training_config['learning_rate']}")
print(f"Weight decay: {training_config['weight_decay']}")
print(f"\nScheduler: Warmup ({warmup_steps} steps) + Cosine Annealing")
print(f"Total training steps: {num_training_steps:,}")
print(f"Warmup steps: {warmup_steps}")
print(f"Cosine steps: {num_training_steps - warmup_steps:,}")
print("="*70)

OPTIMIZER & SCHEDULER
Optimizer: AdamW
Learning rate: 5e-05
Weight decay: 0.01

Scheduler: Warmup (228 steps) + Cosine Annealing
Total training steps: 4,573
Warmup steps: 228
Cosine steps: 4,345


## 9. Training & Validation Functions

In [10]:
from torch.cuda.amp import autocast

def train_epoch(model, train_loader, optimizer, scheduler, epoch, config, writer):
    """Entraîner une epoch."""
    model.train()
    
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch+1}/{config['num_epochs']}"
    )
    
    for step, batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Gérer pixel_values et image_grid_thw (optionnels)
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Forward pass avec mixed precision (bfloat16)
        with autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                labels=labels
            )
            loss = outputs.loss
            
            # Normaliser par gradient accumulation
            loss = loss / config['gradient_accumulation_steps']
        
        # Backward
        loss.backward()
        
        # Update weights chaque gradient_accumulation_steps
        if (step + 1) % config['gradient_accumulation_steps'] == 0:
            # Gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config['max_grad_norm']
            )
            
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # Logging
            global_step = epoch * len(train_loader) + step
            current_lr = scheduler.get_last_lr()[0]
            
            if global_step % config['log_every_n_steps'] == 0:
                writer.add_scalar('train/loss', loss.item() * config['gradient_accumulation_steps'], global_step)
                writer.add_scalar('train/learning_rate', current_lr, global_step)
                writer.add_scalar('train/grad_norm', grad_norm.item(), global_step)
                
                if WANDB_AVAILABLE:
                    wandb.log({
                        'train/loss': loss.item() * config['gradient_accumulation_steps'],
                        'train/learning_rate': current_lr,
                        'train/grad_norm': grad_norm.item(),
                        'epoch': epoch
                    }, step=global_step)
        
        # Accumuler loss
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item() * config['gradient_accumulation_steps']:.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    avg_loss = total_loss / len(train_loader) * config['gradient_accumulation_steps']
    return avg_loss

print("✓ Fonction train_epoch définie")

✓ Fonction train_epoch définie


In [11]:
@torch.no_grad()
def validate(model, val_loader, epoch, config):
    """Valider le modèle avec métriques de classification."""
    model.eval()
    
    total_loss = 0
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(
        val_loader,
        desc=f"Validation Epoch {epoch+1}"
    )
    
    for batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        true_labels = batch['label'].to(device)
        
        # Gérer pixel_values et image_grid_thw (optionnels)
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Forward pass
        with autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                labels=labels
            )
            loss = outputs.loss
        
        total_loss += loss.item()
        
        # Pour les métriques, on collecte les vraies labels
        all_labels.extend(true_labels.cpu().numpy())
        
        progress_bar.set_postfix({'val_loss': f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(val_loader)
    
    # Note: Pour les vraies prédictions, il faudrait générer du texte et parser
    # Pour l'instant, on retourne juste la loss
    # On implémentera la classification complète dans le Notebook 13
    
    return avg_loss

print("✓ Fonction validate définie")

✓ Fonction validate définie


## 10. Setup TensorBoard & WandB

In [12]:
# Déplacer class weights sur le device
class_weights = class_weights.to(device)

# TensorBoard
writer = SummaryWriter(log_dir=LOGS_DIR)
print(f"✓ TensorBoard writer créé: {LOGS_DIR}")
print(f"  → Lancer avec: tensorboard --logdir {LOGS_DIR}")

# WandB (optionnel)
if WANDB_AVAILABLE:
    run_name = f"cassava_vlm_phase2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    wandb.init(
        project="cassava-vlm",
        name=run_name,
        config={
            **training_config,
            'phase': 'phase2_lora',
            'lora_r': lora_config_dict['r'],
            'lora_alpha': lora_config_dict['lora_alpha'],
            'trainable_params': trainable_params,
            'total_params': total_params
        }
    )
    print(f"✓ WandB run initialisé: {run_name}")
else:
    print("⚠️ WandB non disponible, logs TensorBoard uniquement")

✓ TensorBoard writer créé: /home/hounfodjidagba/learning/cassava/outputs/logs/phase2
  → Lancer avec: tensorboard --logdir /home/hounfodjidagba/learning/cassava/outputs/logs/phase2


wandb: Currently logged in as: hospicehounfodji (hospicehounfodji-epitech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB run initialisé: cassava_vlm_phase2_20260101_090435


## 11. Main Training Loop Phase 2

⚠️ **ATTENTION**: Cette cellule va lancer l'entraînement qui durera **~3-6 heures** (2-3 epochs)!

Assurez-vous:
- D'avoir suffisamment de temps
- Que le GPU est disponible
- Que le système ne va pas s'éteindre

In [13]:
print("="*70)
print("DÉBUT DE L'ENTRAÎNEMENT PHASE 2")
print("="*70)
print(f"Durée estimée: ~3-6 heures (2-3 epochs)")
print(f"Max epochs: {training_config['num_epochs']}")
print(f"Early stopping patience: {training_config['early_stopping_patience']}")
print(f"Train batches: {len(train_loader):,}")
print(f"Effective batch size: {training_config['batch_size'] * training_config['gradient_accumulation_steps']}")
print("="*70)

start_time = time.time()
best_val_loss = float('inf')

# Initialiser Early Stopping
early_stopping = EarlyStopping(
    patience=training_config['early_stopping_patience'],
    threshold=training_config['early_stopping_threshold'],
    verbose=True
)

# Training loop
for epoch in range(training_config['num_epochs']):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch + 1}/{training_config['num_epochs']}")
    print(f"{'='*70}")
    
    # Training
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler,
        epoch, training_config, writer
    )
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    
    # Validation
    val_loss = validate(model, val_loader, epoch, training_config)
    print(f"Val Loss: {val_loss:.4f}")
    
    # Log epoch metrics
    writer.add_scalar('epoch/train_loss', train_loss, epoch)
    writer.add_scalar('epoch/val_loss', val_loss, epoch)
    
    if WANDB_AVAILABLE:
        wandb.log({
            'epoch/train_loss': train_loss,
            'epoch/val_loss': val_loss,
            'epoch': epoch
        })
    
    # Early stopping check
    early_stopping(val_loss, epoch)
    
    # Save checkpoint
    checkpoint_path = CHECKPOINT_DIR / f"epoch_{epoch+1}"
    checkpoint_path.mkdir(exist_ok=True)
    
    model.save_pretrained(checkpoint_path)
    processor.save_pretrained(checkpoint_path)
    
    # Save training state
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'config': training_config,
        'early_stopping_state': {
            'counter': early_stopping.counter,
            'best_loss': early_stopping.best_loss,
            'best_epoch': early_stopping.best_epoch
        }
    }, checkpoint_path / 'training_state.pt')
    
    print(f"\n✓ Checkpoint sauvegardé: {checkpoint_path}")
    
    # Best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_checkpoint_path = CHECKPOINT_DIR / "best"
        best_checkpoint_path.mkdir(exist_ok=True)
        
        model.save_pretrained(best_checkpoint_path)
        processor.save_pretrained(best_checkpoint_path)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'config': training_config
        }, best_checkpoint_path / 'training_state.pt')
        
        print(f"✓ Best checkpoint sauvegardé: {best_checkpoint_path}")
    
    # Check early stopping
    if early_stopping.early_stop:
        print(f"\n⛔ Early stopping déclenché à l'epoch {epoch + 1}")
        print(f"   Meilleur modèle: epoch {early_stopping.best_epoch + 1} (val_loss: {early_stopping.best_loss:.4f})")
        break

end_time = time.time()
training_duration = end_time - start_time

print("\n" + "="*70)
print("ENTRAÎNEMENT PHASE 2 TERMINÉ")
print("="*70)
print(f"Durée totale: {training_duration / 3600:.2f}h")
print(f"Epochs complétées: {epoch + 1}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best checkpoint: {best_checkpoint_path}")
print("="*70)

# Sauvegarder les métriques finales
final_metrics = {
    'phase': 'phase2_lora',
    'total_epochs': epoch + 1,
    'training_duration_hours': training_duration / 3600,
    'best_val_loss': best_val_loss,
    'best_epoch': early_stopping.best_epoch + 1,
    'early_stopped': early_stopping.early_stop,
    'config': training_config
}

metrics_path = METRICS_DIR / 'phase2_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(final_metrics, f, indent=2)

print(f"\n✓ Métriques sauvegardées: {metrics_path}")

# Fermer les loggers
writer.close()
if WANDB_AVAILABLE:
    wandb.finish()

print("\n✓ Logs fermés")

DÉBUT DE L'ENTRAÎNEMENT PHASE 2
Durée estimée: ~3-6 heures (2-3 epochs)
Max epochs: 3
Early stopping patience: 2
Train batches: 12,196
Effective batch size: 16

EPOCH 1/3


Epoch 1/3:   0%|          | 0/12196 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...



Train Loss: 6.4089


Validation Epoch 1:   0%|          | 0/1525 [00:00<?, ?it/s]

...


Train Loss: 6.0044


Validation Epoch 3:   0%|          | 0/1525 [00:00<?, ?it/s]

Val Loss: 6.0045
⚠️ Early Stopping: No improvement (2/2)
⛔ Early Stopping: Triggered! Best was epoch 1 with loss 6.0054

✓ Checkpoint sauvegardé: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase2_lora/epoch_3
✓ Best checkpoint sauvegardé: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase2_lora/best

⛔ Early stopping déclenché à l'epoch 3
   Meilleur modèle: epoch 1 (val_loss: 6.0054)

ENTRAÎNEMENT PHASE 2 TERMINÉ
Durée totale: 3.54h
Epochs complétées: 3
Best val loss: 6.0045
Best checkpoint: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase2_lora/best

✓ Métriques sauvegardées: /home/hounfodjidagba/learning/cassava/outputs/phase2_metrics.json


epoch,▁▅█
epoch/train_loss,█▁▁
epoch/val_loss,█▃▁
epoch,2
epoch/train_loss,6.00436
epoch/val_loss,6.00446



✓ Logs fermés


## 12. Résumé et Prochaines Étapes

### ✓ Accompli dans ce notebook:
1. **Checkpoint Phase 1 chargé** - Projecteur aligné
2. **LoRA adapters dégelés** - Fine-tuning complet
3. **Training 2-3 epochs** - Avec early stopping
4. **Monitoring** - TensorBoard & WandB logs complets
5. **Checkpoints** - Sauvegardés par epoch + best

### 🎯 Résultats attendus:
- **Val loss**: Amélioration significative vs Phase 1
- **Convergence**: Early stopping après 2-3 epochs
- **Prêt pour évaluation**: Notebook 13

### 📋 Prochaines étapes (Notebook 13 - Évaluation):
1. Charger best checkpoint Phase 2
2. Évaluation complète sur test set
3. Métriques de classification (accuracy, F1, confusion matrix)
4. Génération qualitative (exemples de descriptions)
5. Comparaison baseline → Phase 1 → Phase 2
6. Target: 95-97% accuracy

### 📊 Fichiers générés:
- `models/checkpoints/phase2_lora/epoch_*/` - Checkpoints par epoch
- `models/checkpoints/phase2_lora/best/` - Best checkpoint
- `outputs/logs/phase2/` - TensorBoard logs
- `outputs/phase2_metrics.json` - Métriques finales
- WandB dashboard online (si disponible)